# 07 - Green Hydrogen Valley & Cross-Sector Coupling

In this notebook, we model an integrated regional **Green Hydrogen Valley**:

1. Hybrid **PEM + Alkaline + SOEC** electrolyzer park.
2. Waste heat recovery from electrolysis routed into district heating via an **Industrial Heat Pump**.
3. High-pressure **H2 Compressor** and **350-bar Storage Tank**.
4. **Methanation Reactor** (Power-to-Gas) combining green $\text{H}_2$ with captured $\text{CO}_2$ to produce synthetic natural gas (SNG).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world
from energysim.models.hydrogen import (
    PEMElectrolyzer, AlkalineElectrolyzer, SOECElectrolyzer,
    H2Compressor, H2StorageTank, MethanationReactor
)
from energysim.models.hvac_thermal import IndustrialHeatPump

print("Hydrogen Valley models loaded!")


## Step 1: Instantiate the Sector-Coupled Hydrogen Park


In [ ]:
sim = world(start_time=0, stop_time=86400, t_macro=900)

# Electrolyzers
pem = PEMElectrolyzer(name="pem_stack", p_nominal_mw=10.0)
ael = AlkalineElectrolyzer(name="ael_stack", p_nominal_mw=20.0, min_turndown_pct=20.0)

# Balance of Plant
comp = H2Compressor(name="compressor", p_in_bar=30.0, p_out_bar=350.0)
tank = H2StorageTank(name="storage", volume_m3=200.0, p_max_bar=350.0, p_init_bar=80.0)

# Waste heat recovery
hp = IndustrialHeatPump(name="district_hp", p_electric_max_mw=3.0, t_source_nom_c=45.0, t_sink_nom_c=90.0)

# Methanation
p2g = MethanationReactor(name="methanation", capacity_mw_sng=5.0)

# Add to canvas
sim.add_model("pem", model=pem, outputs=["P_actual", "m_dot_H2_kg_hr", "Q_waste_MW"])
sim.add_model("ael", model=ael, outputs=["P_actual", "m_dot_H2_kg_hr"])
sim.add_model("compressor", model=comp, outputs=["P_electric_MW", "m_dot_out_kg_hr"])
sim.add_model("tank", model=tank, outputs=["pressure_bar", "mass_stored_kg"])
sim.add_model("heat_pump", model=hp, outputs=["P_actual", "Q_heat_MW", "COP"])
sim.add_model("methanation", model=p2g, outputs=["m_dot_CH4_kg_hr", "Q_exothermic_MWth"])


## Step 2: Establish Cross-Carrier Couplings


In [ ]:
# Route electrolyzer hydrogen to compressor, and compressor discharge into buffer storage
sim.add_connections({
    "pem.m_dot_H2_kg_hr": "compressor.m_dot_in_kg_hr",
    "compressor.m_dot_out_kg_hr": "tank.m_dot_in_kg_hr",
})

# Power dispatch orders
sim.add_signal("wind_surplus", lambda t: 8.0)
sim.add_connections({"wind_surplus.y": "pem.P_cmd"})

print("Sector-coupled topology connected!")


## Step 3: Simulate and Extract Hydrogen Valley Metrics


In [ ]:
sim.simulate(dashboard=False)
results = sim.results(dashboard=False)
display(results["tank"].tail())
